In [ ]:
import sys, os, warnings
from pathlib import Path
import torch

NOTEBOOK_DIR = Path().resolve()
SRC_DIR      = NOTEBOOK_DIR.parent / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from dataloaders._ts_dataloader import DataLoaderFactory
from common.train import train, eval_test
from models.linear import TinyLinearModel
from common._utils import _dataset_entry
from _test_utils import make_df, make_mcfg, make_entry, make_dcfg, setup_test_data, remove_test_data

TEST_DATA_DIR = os.path.join(os.getcwd(), "test_data")

warnings.filterwarnings("ignore")
print("imports OK")

## Test: HorizonBatchSampler - is_multivariate always False

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    p = os.path.join(TEST_DATA_DIR, "test.csv")
    make_df(n_series=1).to_csv(p, index=False)

    mcfg = make_mcfg(model_name='linear')
    dcfg = make_dcfg([make_entry(p, name="test", multivariate=False)])
    factory = DataLoaderFactory(mcfg, dcfg)
    loader  = factory.train_dataloader()

    failed = [b["is_multivariate"] for b in loader if b["is_multivariate"] is not False]
    if len(failed) == 0:
        print("\n✓ TEST PASSED")
finally:
    remove_test_data(TEST_DATA_DIR)

## Test: HorizonBatchSampler - is_multivariate always True

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    p = os.path.join(TEST_DATA_DIR, "test.csv")
    make_df(n_series=5).to_csv(p, index=False)
 
    mcfg = make_mcfg(model_name='linear')
    dcfg = make_dcfg([make_entry(p, name="test", multivariate=True)])
    factory = DataLoaderFactory(mcfg, dcfg)
    loader = factory.train_dataloader()
 
    failed = [b["is_multivariate"] for b in loader if b["is_multivariate"] is not True]
    if len(failed) == 0:
        print("\n✓ TEST PASSED")
finally:
    remove_test_data(TEST_DATA_DIR)

## Test: HorizonBatchSampler - no batch contains both univariate and multivariate data

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    p_uni = os.path.join(TEST_DATA_DIR, "test_uni.csv")
    p_mv = os.path.join(TEST_DATA_DIR, "test_mv.csv")
    make_df(n_series=1).to_csv(p_uni, index=False)
    make_df(n_series=5).to_csv(p_mv,  index=False)
 
    mcfg = make_mcfg()
    dcfg = make_dcfg([
        make_entry(p_uni, name="test_uni", multivariate=False),
        make_entry(p_mv,  name="test_mv",  multivariate=True),
    ])
    factory = DataLoaderFactory(mcfg, dcfg)
    loader  = factory.train_dataloader()
 
    mixed = []
    for batch in loader:
        flag  = batch["is_multivariate"]
        names = batch["dataset_name"]
        if any((name == "test_mv") != flag for name in names):
            mixed.append(names)
 
    if len(mixed)==0:
        print("\n✓ TEST PASSED")
finally:
    remove_test_data(TEST_DATA_DIR)

## Test: HorizonBatchSampler: same horizon wth mixed dataset batch

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    p_uni = os.path.join(TEST_DATA_DIR, "test_uni.csv")
    p_mv = os.path.join(TEST_DATA_DIR, "test_mv.csv")
    make_df(n_series=1).to_csv(p_uni, index=False)
    make_df(n_series=5).to_csv(p_mv,  index=False)
 
    mcfg = make_mcfg(h=6)
    dcfg = make_dcfg([
        make_entry(p_uni, name="test_uni", horizon=6, multivariate=False),
        make_entry(p_mv,  name="test_mv",  horizon=6, multivariate=True),
    ])
    factory = DataLoaderFactory(mcfg, dcfg)
    loader  = factory.train_dataloader()
 
    seen_groups = set()
    mixed       = []
    for batch in loader:
        flag  = batch["is_multivariate"]
        names = batch["dataset_name"]
        seen_groups.add((int(batch["horizon"][0]), flag))
        if any((name == "test_mv") != flag for name in names):
            mixed.append(names)
 
    if (len(mixed) == 0) and ({(6, False), (6, True)}.issubset(seen_groups)):
        print("\n✓ TEST PASSED")

finally:
    remove_test_data(TEST_DATA_DIR)

## Test: HorizonBatchSampler: separation holds across multiple epochs

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    p_uni = os.path.join(TEST_DATA_DIR, "test_uni.csv")
    p_mv = os.path.join(TEST_DATA_DIR, "test_mv.csv")
    make_df(n_series=1).to_csv(p_uni, index=False)
    make_df(n_series=5).to_csv(p_mv,  index=False)
 
    mcfg = make_mcfg()
    dcfg = make_dcfg([
        make_entry(p_uni, name="test_uni", multivariate=False),
        make_entry(p_mv,  name="test_mv",  multivariate=True),
    ])
    factory = DataLoaderFactory(mcfg, dcfg)
    loader  = factory.train_dataloader()
    sampler = loader.batch_sampler
 
    failures = []
    for epoch in range(5):
        sampler.set_epoch(epoch)
        for batch in loader:
            flag  = batch["is_multivariate"]
            names = batch["dataset_name"]
            if any((name == "test_mv") != flag for name in names):
                failures.append((epoch, names))
 
    if len(failures) == 0:
        print("\n✓ TEST PASSED")
finally:
    remove_test_data(TEST_DATA_DIR)

## Test: HorizonBatchSampler — `train()`, single dataset (default)

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    p = os.path.join(TEST_DATA_DIR, "test.csv")
    df = make_df(n_series=3, n_steps=300)
    df.to_csv(p, index=False)

    mcfg = make_mcfg(
        model_name     ='linear',
        context_length = 32,
        fcd_samples    = 4,
        batch_size     = 2,
        max_steps      = 20,
        mixing_strategy= "concat",
        checkpoint_dir = TEST_DATA_DIR,
    )
    entry = make_entry(p, name="test", horizon=6, val_size=50, test_size=50)
    dcfg  = make_dcfg([entry])

    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()

    model = TinyLinearModel(mcfg)
    metrics = train(model, mcfg, train_loader, val_loaders,
                    device=torch.device("cpu"), seed=0)

    print(f"\nfinal val metrics: {metrics}")
    assert Path(TEST_DATA_DIR, "final.pt").exists(), "final.pt not saved"

    # Check predictions
    test_loaders = factory.test_dataloaders()
    preds = eval_test(model, factory, device=torch.device("cpu"))
    assert "test" in preds, "missing test predictions"
    print("\n✓ TEST PASSED")
finally:
    remove_test_data(TEST_DATA_DIR)

## Test: HorizonBatchSampler — `train()`, two datasets, weighted concat mixing

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    p_a = os.path.join(TEST_DATA_DIR, "testa.csv")
    p_b = p = os.path.join(TEST_DATA_DIR, "testb.csv")
    make_df(n_series=3, n_steps=300, seed=1).to_csv(p_a, index=False)
    make_df(n_series=2, n_steps=300, seed=2).to_csv(p_b, index=False)

    mcfg = make_mcfg(
        model_name      ='linear',
        context_length  = 32,
        fcd_samples     = 4,
        batch_size      = 2,
        max_steps       = 20,
        mixing_strategy = "concat",
        checkpoint_dir  = TEST_DATA_DIR,
    )

    # Same horizon, different weights — dataset A gets 3x more samples than B
    entry_a = make_entry(p_a, name="ds_a", horizon=6, val_size=50, test_size=50, weight=3.0)
    entry_b = make_entry(p_b, name="ds_b", horizon=6, val_size=50, test_size=50, weight=1.0)
    dcfg    = make_dcfg([entry_a, entry_b])

    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()

    # Max channels across both datasets = 3 (ds_a)
    model   = TinyLinearModel(mcfg)
    metrics = train(model, mcfg, train_loader, val_loaders,
                    device=torch.device("cpu"), seed=0)

    print(f"final val metrics: {metrics}")

    # Verify both datasets appear in val/test
    preds = eval_test(model, factory, device=torch.device("cpu"))
    assert "ds_a" in preds and "ds_b" in preds, f"missing keys in preds: {preds.keys()}"
    print("\n✓ TEST PASSED")
finally:
    remove_test_data(TEST_DATA_DIR)

## Test: BatchSampler — univariate/multivariate never mixed

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    p_uni = os.path.join(TEST_DATA_DIR, "test_uni.csv")
    p_mv  = os.path.join(TEST_DATA_DIR, "test_mv.csv")
    make_df(n_series=1).to_csv(p_uni, index=False)
    make_df(n_series=5).to_csv(p_mv,  index=False)
    mcfg = make_mcfg(batch_sampler="flat", horizon_override=6)
    dcfg = make_dcfg([
        make_entry(p_uni, name="test_uni", horizon=6,  multivariate=False),
        make_entry(p_mv,  name="test_mv",  horizon=12, multivariate=True),
    ])
    factory = DataLoaderFactory(mcfg, dcfg)
    loader  = factory.train_dataloader()
    mixed = []
    for batch in loader:
        flag  = batch["is_multivariate"]
        names = batch["dataset_name"]
        if any((name == "test_mv") != flag for name in names):
            mixed.append(names)
    if len(mixed) == 0:
        print("\n✓ TEST PASSED")
finally:
    remove_test_data(TEST_DATA_DIR)

## Test: BatchSampler — horizons mixed across univariate batches

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    p_h6  = os.path.join(TEST_DATA_DIR, "test_h6.csv")
    p_h12 = os.path.join(TEST_DATA_DIR, "test_h12.csv")
    make_df(n_series=1).to_csv(p_h6,  index=False)
    make_df(n_series=1).to_csv(p_h12, index=False)
    mcfg = make_mcfg(batch_sampler="flat", horizon_override=6)
    dcfg = make_dcfg([
        make_entry(p_h6,  name="test_h6",  horizon=6,  multivariate=False),
        make_entry(p_h12, name="test_h12", horizon=12, multivariate=False),
    ])
    factory       = DataLoaderFactory(mcfg, dcfg)
    loader        = factory.train_dataloader()
    horizons_seen = set()
    for batch in loader:
        for h in batch["horizon"].tolist():
            horizons_seen.add(h)
    if len(horizons_seen) > 1:
        print(f"\n✓ TEST PASSED — horizons mixed across batches {horizons_seen}")
finally:
    remove_test_data(TEST_DATA_DIR)

## Test: BatchSampler — epoch reproducibility

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    p_uni = os.path.join(TEST_DATA_DIR, "test_uni.csv")
    p_mv  = os.path.join(TEST_DATA_DIR, "test_mv.csv")
    make_df(n_series=1).to_csv(p_uni, index=False)
    make_df(n_series=5).to_csv(p_mv,  index=False)
    mcfg = make_mcfg(batch_sampler="flat", horizon_override=6)
    dcfg = make_dcfg([
        make_entry(p_uni, name="test_uni", horizon=6,  multivariate=False),
        make_entry(p_mv,  name="test_mv",  horizon=6,  multivariate=True),
    ])
    factory = DataLoaderFactory(mcfg, dcfg)
    loader  = factory.train_dataloader()
    sampler = loader.batch_sampler
    failures = []
    epoch_batches = {}
    for epoch in [0, 1, 0]:
        sampler.set_epoch(epoch)
        names_seq = [batch["dataset_name"] for batch in loader]
        if epoch not in epoch_batches:
            epoch_batches[epoch] = names_seq
        elif epoch_batches[epoch] != names_seq:
            failures.append(f"epoch {epoch} not reproducible")
    if epoch_batches[0] == epoch_batches[1]:
        failures.append("epoch 0 and epoch 1 produced identical order")
    if len(failures) == 0:
        print("\n✓ TEST PASSED")
finally:
    remove_test_data(TEST_DATA_DIR)